# What an honest Grover oracle costs

Two open questions from the nonogram study, in one notebook. It is self-contained: it needs
`qiskit`, `qiskit-algorithms` and `qiskit-ibm-runtime`, and nothing from the project repo.

**Question 1 — scaling.** At 3x3 an oracle that tests the clues costs about 4.3x one that
marks grids already known to satisfy them. Three shapes is not a trend. Does the ratio hold
as the board grows?

**Question 2 — spread.** The clue oracle's cost depends on which patterns the clues allow,
so unlike the answer-marking form it differs board to board. Only three boards per class
have ever been measured. What does the distribution look like?

**The two oracles.** Qiskit's `PhaseOracleGate` reduces the clue formula over its whole
`2^n` truth table and emits one multi-controlled Z per solution: its cost tracks the number
of solutions, and building it means solving the puzzle first. The clue oracle here carries
one flag qubit per line, raises it with one multi-controlled X per pattern that line's clue
allows, flips the phase on all flags at once, then lowers the flags again. It never consults
a solution.

**Cost to run.** The answer-marking side is the expensive one: its synthesis doubles per
cell, measured at 44 s for 18 cells on a 2019 laptop, so roughly 12 minutes at 22 cells and
about 3 hours at 25. `SKIP_MARKING_ABOVE` in the config stops measuring it past a size you
choose; the clue oracle alone stays affordable much further.

## 1. Environment

In [ ]:
# pip install qiskit qiskit-algorithms qiskit-ibm-runtime
import os

# Qiskit's own threading competes with the process pool below for the same cores, and one
# circuit per worker is the coarser and faster split.
os.environ.setdefault("QISKIT_NUM_PROCS", "1")
os.environ.setdefault("RAYON_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import qiskit, qiskit_algorithms, qiskit_ibm_runtime
from qiskit_ibm_runtime.fake_provider import FakeTorino

print("qiskit", qiskit.__version__)
print("qiskit-algorithms", qiskit_algorithms.__version__)
print("qiskit-ibm-runtime", qiskit_ibm_runtime.__version__)
print("cores", os.cpu_count())
print("target", FakeTorino().name, FakeTorino().num_qubits, "qubits")

## 2. Configuration

The knobs worth touching are all here. `seeds = 8` matches the committed cost table; 4 is
about twice as fast and gives up a little depth. `device = False` skips routing onto the
lattice and measures synthesis alone, which is several times quicker and enough if you only
want the ratio between the two oracles.

`workers` defaults to a quarter of the cores, to 16 at most. This is a shared machine, and
each worker also drops its own scheduling priority, so the sweep yields to anything else
running rather than competing with it. Raise the number if the box is yours.

In [ ]:
CONFIG = {
    "seeds": 8,             # transpiler seeds per board; the cost table used 8
    "spare": 2,             # idle qubits lent to synthesis for the big gates
    "device": True,         # False measures synthesis alone, with no coupling map
    # A quarter of the cores, capped: this runs on a machine other people use, and
    # Windows refuses a pool wider than 61 whatever the core count. Raise it if the
    # box is yours for the evening.
    "workers": max(1, min((os.cpu_count() or 4) // 4, 16)),
    "timeout": 3600,        # seconds per board, 0 for none
    "skip_marking_above": 16,   # cells past which Qiskit's oracle is not measured
    "sample_seed": 20260930,
}

# Question 1: one board per (shape, solution count), smallest shapes first.
SCALING = {"min_cells": 4, "max_cells": 20, "max_solutions": 4, "counts_per_shape": 3}

# Question 2: every solvable board of a shape, or a sample when the space is too large.
SPREAD = {"shape": (4, 4), "sample": 2000}

OUT_SCALING = "clue-oracle-scaling.jsonl"
OUT_SPREAD = "clue-oracle-spread.jsonl"
CONFIG

## 3. The measurement code, as a module the workers can import

Windows and macOS start a worker with `spawn` rather than `fork`, and a spawned process can
import a module but cannot reach functions defined in a notebook cell. So the compute lives
in `clue_oracle_worker.py`, written beside this notebook by the cell below and imported
back. Re-run the cell after editing it.

In [ ]:
MODULE_SRC = """
'''Everything a worker process needs, in a module it can import.

Written beside the notebook rather than defined in it: on Windows and macOS a new process
is started with `spawn` rather than `fork`, and a spawned child can import a module but
cannot reach functions defined in a notebook cell.
'''
from __future__ import annotations

import itertools
import math
import os
import random
import signal
import time
from dataclasses import dataclass, asdict

os.environ.setdefault("QISKIT_NUM_PROCS", "1")
os.environ.setdefault("RAYON_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

#: Unix can cap a board with an alarm; Windows cannot, and says so rather than pretending.
CAN_TIME_OUT = hasattr(signal, "SIGALRM")


def _step_aside():
    '''Run behind whatever else the machine is doing.

    A transpile sweep is nobody else's priority. Lowering it costs this run very little on
    an idle box and keeps it out of the way on a busy one.
    '''
    try:
        if hasattr(os, "nice"):
            os.nice(10)
        elif os.name == "nt":
            import ctypes

            BELOW_NORMAL_PRIORITY_CLASS = 0x00004000
            handle = ctypes.windll.kernel32.GetCurrentProcess()
            ctypes.windll.kernel32.SetPriorityClass(handle, BELOW_NORMAL_PRIORITY_CLASS)
    except Exception:
        pass  # a scheduler that refuses is not a reason to stop measuring


_step_aside()


def line_patterns(length, clue):
    '''Every filling of one line that matches its clue, as ints with bit 0 leftmost.'''
    if clue in ((0,), ()):
        return [0]
    out = []

    def place(block, start, bits):
        if block == len(clue):
            out.append(bits)
            return
        run = clue[block]
        rest = clue[block + 1:]
        room = sum(rest) + len(rest)
        for pos in range(start, length - run - room + 1):
            mask = 0
            for b in range(run):
                mask |= 1 << (pos + b)
            place(block + 1, pos + run + 1, bits | mask)

    place(0, 0, 0)
    return out


def line_clue(cells):
    '''The clue describing one line of filled and empty cells.'''
    out, run = [], 0
    for cell in cells:
        if cell:
            run += 1
        elif run:
            out.append(run)
            run = 0
    if run:
        out.append(run)
    return tuple(out) if out else (0,)


def clues_of(bits, rows, cols):
    '''The row and column clues a filled grid produces.'''
    grid = [bits[r * cols:(r + 1) * cols] for r in range(rows)]
    return ([line_clue(row) for row in grid],
            [line_clue([grid[r][c] for r in range(rows)]) for c in range(cols)])


def solve(row_clues, col_clues, limit=64):
    '''Every grid the clues accept, by placing a row at a time and pruning the columns.'''
    rows, cols = len(row_clues), len(col_clues)
    row_options = [line_patterns(cols, c) for c in row_clues]
    col_options = [line_patterns(rows, c) for c in col_clues]
    if any(not o for o in row_options) or any(not o for o in col_options):
        return []

    found = []

    def walk(r, placed, col_live):
        if len(found) >= limit:
            return
        if r == rows:
            found.append(list(placed))
            return
        for pattern in row_options[r]:
            nxt = []
            for c in range(cols):
                bit = (pattern >> c) & 1
                live = [p for p in col_live[c] if ((p >> r) & 1) == bit]
                if not live:
                    break
                nxt.append(live)
            else:
                placed.append(pattern)
                walk(r + 1, placed, nxt)
                placed.pop()

    walk(0, [], col_options)
    return found


def boards(rows, cols, sample=None, seed=0):
    '''Distinct solvable puzzles of this shape, with their solution counts.'''
    cells = rows * cols
    seen = set()

    def emit(bits):
        row_clues, col_clues = clues_of(bits, rows, cols)
        key = (tuple(row_clues), tuple(col_clues))
        if key in seen:
            return None
        seen.add(key)
        answers = solve(row_clues, col_clues)
        return (row_clues, col_clues, len(answers)) if answers else None

    if sample is None and cells <= 20:
        for bits in itertools.product([0, 1], repeat=cells):
            got = emit(list(bits))
            if got:
                yield got
        return

    rng = random.Random(seed)
    for _ in range(sample or 2000):
        got = emit([rng.randint(0, 1) for _ in range(cells)])
        if got:
            yield got


def optimal_rounds(solutions, cells):
    '''Grover iteration count, as the backend and the page both compute it.'''
    if solutions <= 0:
        return 0
    return max(1, math.floor(math.pi / 4 * math.sqrt(2 ** cells / solutions)))


def clue_oracle(row_clues, col_clues):
    '''A phase oracle that tests the clues: one flag qubit per line.

    For each line, one multi-controlled X per pattern that line's clue allows raises its
    flag; the patterns are mutually exclusive, so toggling acts as an OR. A phase flip
    conditioned on every flag marks the grids satisfying all of them. Repeating the pattern
    gates lowers the flags again -- left raised they stay entangled with the cells and the
    interference Grover depends on falls apart.
    '''
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import MCXGate, ZGate

    rows, cols = len(row_clues), len(col_clues)
    cells = rows * cols
    lines = [([r * cols + c for c in range(cols)], tuple(row_clues[r])) for r in range(rows)]
    lines += [([r * cols + c for r in range(rows)], tuple(col_clues[c])) for c in range(cols)]

    qc = QuantumCircuit(cells + len(lines), name="clue oracle")

    def mark():
        for flag, (qubits, clue) in enumerate(lines):
            for pattern in line_patterns(len(qubits), clue):
                qc.append(MCXGate(len(qubits), ctrl_state=pattern), qubits + [cells + flag])

    mark()
    qc.append(ZGate().control(len(lines) - 1), [cells + i for i in range(len(lines))])
    mark()
    return qc


def clue_grover(row_clues, col_clues, rounds, spare):
    '''Grover over the clue oracle, with spare qubits for synthesis to borrow.'''
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import ZGate

    oracle = clue_oracle(row_clues, col_clues)
    cells = len(row_clues) * len(col_clues)
    qc = QuantumCircuit(oracle.num_qubits + spare, cells)
    qc.h(range(cells))
    for _ in range(rounds):
        qc.compose(oracle, qubits=range(oracle.num_qubits), inplace=True)
        qc.h(range(cells))
        qc.x(range(cells))
        qc.append(ZGate().control(cells - 1), list(range(cells)))
        qc.x(range(cells))
        qc.h(range(cells))
    qc.measure(range(cells), range(cells))
    return qc


def clue_formula(row_clues, col_clues):
    '''The clues as a boolean expression, which is what Qiskit's oracle is built from.'''
    rows, cols = len(row_clues), len(col_clues)
    lines = [([r * cols + c for c in range(cols)], tuple(row_clues[r])) for r in range(rows)]
    lines += [([r * cols + c for r in range(rows)], tuple(col_clues[c])) for c in range(cols)]
    terms = []
    for qubits, clue in lines:
        options = []
        for pattern in line_patterns(len(qubits), clue):
            literals = [("v%d" % q if (pattern >> i) & 1 else "~v%d" % q)
                        for i, q in enumerate(qubits)]
            options.append("(" + " & ".join(literals) + ")")
        terms.append("(" + " | ".join(options) + ")")
    return " & ".join(terms)


def marking_grover(row_clues, col_clues, rounds, spare):
    '''Grover over Qiskit's oracle, which compiles to one marked grid per solution.'''
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import PhaseOracleGate
    from qiskit_algorithms import AmplificationProblem, Grover

    oracle = PhaseOracleGate(clue_formula(row_clues, col_clues))
    base = Grover(iterations=rounds).construct_circuit(
        AmplificationProblem(oracle), measurement=True)
    if not spare:
        return base
    wide = QuantumCircuit(base.num_qubits + spare, base.num_clbits)
    wide.compose(base, qubits=range(base.num_qubits), clbits=range(base.num_clbits),
                 inplace=True)
    return wide


@dataclass
class Cost:
    two_qubit: int
    depth: int
    gates: int
    qubits: int
    seconds: float


class Timeout(Exception):
    pass


def _alarm(signum, frame):
    raise Timeout()


def transpile_cost(circuit, device, seeds):
    '''Transpile at several seeds and keep the shallowest, as the cost table does.'''
    from qiskit import transpile

    basis = ["cz", "rz", "sx", "x", "id"]
    backend = None
    if device:
        from qiskit_ibm_runtime.fake_provider import FakeTorino
        backend = FakeTorino()

    started = time.time()
    best = None
    for seed in range(seeds):
        if backend is not None:
            flat = transpile(circuit, backend=backend, optimization_level=3,
                             seed_transpiler=seed)
        else:
            flat = transpile(circuit, basis_gates=basis, coupling_map=None,
                             optimization_level=3, seed_transpiler=seed)
        ops = flat.count_ops()
        two_qubit = sum(v for g, v in ops.items() if g in ("ecr", "cz", "cx"))
        found = Cost(two_qubit, flat.depth(), sum(ops.values()), flat.num_qubits, 0.0)
        if best is None or found.depth < best.depth:
            best = found
    best.seconds = round(time.time() - started, 2)
    return best


def measure_board(job):
    '''One board, one or both oracles. Returns a plain dict, safe to append as JSON.'''
    key, row_clues, col_clues, solutions, cfg = job
    row_clues = [tuple(c) for c in row_clues]
    col_clues = [tuple(c) for c in col_clues]
    rows, cols = len(row_clues), len(col_clues)
    cells = rows * cols
    rounds = optimal_rounds(solutions, cells)
    out = {"key": key, "rows": rows, "cols": cols, "cells": cells, "solutions": solutions,
           "rounds": rounds, "device": cfg["device"], "seeds": cfg["seeds"],
           "spare": cfg["spare"],
           "row_clues": [list(c) for c in row_clues],
           "col_clues": [list(c) for c in col_clues]}

    capped = bool(cfg["timeout"]) and CAN_TIME_OUT
    if capped:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(cfg["timeout"])
    try:
        clue = transpile_cost(clue_grover(row_clues, col_clues, rounds, cfg["spare"]),
                              cfg["device"], cfg["seeds"])
        out["clue"] = asdict(clue)
        if cells <= cfg["skip_marking_above"]:
            marking = transpile_cost(
                marking_grover(row_clues, col_clues, rounds, cfg["spare"]),
                cfg["device"], cfg["seeds"])
            out["marking"] = asdict(marking)
            out["ratio_two_qubit"] = round(clue.two_qubit / marking.two_qubit, 3)
            out["ratio_depth"] = round(clue.depth / marking.depth, 3)
    except Timeout:
        out["timeout"] = cfg["timeout"]
    except Exception as exc:
        out["error"] = "%s: %s" % (type(exc).__name__, exc)
    finally:
        if capped:
            signal.alarm(0)
    return out
"""

from pathlib import Path

WORKER = Path("clue_oracle_worker.py")
WORKER.write_text(MODULE_SRC, encoding="utf-8")

import importlib, sys
sys.path.insert(0, str(WORKER.resolve().parent))
import clue_oracle_worker as W
W = importlib.reload(W)

line_patterns, line_clue, clues_of = W.line_patterns, W.line_clue, W.clues_of
solve, boards, optimal_rounds = W.solve, W.boards, W.optimal_rounds
clue_oracle, clue_grover, marking_grover = W.clue_oracle, W.clue_grover, W.marking_grover

print("wrote", WORKER.resolve())
print("3x3 plus solves to", len(solve([(1,), (3,), (1,)], [(1,), (3,), (1,)])), "grid")
print("3x3 clue oracle:", clue_oracle([(1,), (3,), (1,)], [(1,), (3,), (1,)]).num_qubits, "qubits")
print("per-board timeout available on this platform:", W.CAN_TIME_OUT)

## 5. Check the oracle before spending machine time

The clue oracle is only worth measuring if it is correct. After the ideal number of rounds
the solutions should carry the textbook probability `sin((2k+1)theta)^2`. An oracle marking
the wrong set, or leaving a flag entangled with the cells, misses this by far more than the
tolerance. Run this first -- it takes seconds.

In [ ]:
import math

from qiskit.quantum_info import Statevector

def check(row_clues, col_clues):
    cells = len(row_clues) * len(col_clues)
    answers = W.solve(row_clues, col_clues)
    marked = len(answers)
    rounds = W.optimal_rounds(marked, cells)

    qc = W.clue_grover(row_clues, col_clues, rounds, 0).remove_final_measurements(inplace=False)
    probs = Statevector(qc).probabilities(range(cells))
    # A solution is a list of row patterns; cell i of the grid is qubit i.
    hit = 0.0
    for answer in answers:
        index = 0
        for r, pattern in enumerate(answer):
            for c in range(len(col_clues)):
                if (pattern >> c) & 1:
                    index |= 1 << (r * len(col_clues) + c)
        hit += probs[index]

    theta = math.asin(math.sqrt(marked / 2 ** cells))
    ideal = math.sin((2 * rounds + 1) * theta) ** 2
    print("%s cells, M=%d, k=%2d: measured %.6f  textbook %.6f  delta %.1e"
          % (cells, marked, rounds, hit, ideal, abs(hit - ideal)))
    return abs(hit - ideal)

worst = max(
    check([(1,), (1,)], [(1,), (1,)]),
    check([(1,), (1,)], [(1,), (0,), (1,)]),
    check([(1,), (3,), (1,)], [(1,), (3,), (1,)]),
)
assert worst < 1e-9, "the oracle does not amplify what the clues accept"
print("\noracle verified")

## 6. Measurement

Results are appended as JSON Lines and flushed per board, and a rerun skips keys already in
the file -- so this can be interrupted, resumed, or extended to larger shapes without
recomputing. Any board that fails to compile records the error and the sweep carries on.

Work is spread with a process pool on every platform, because the compute lives in the
module written above rather than in a notebook cell. The per-board timeout needs `SIGALRM`
and so applies on Unix only; on Windows the sweep says so and relies on
`skip_marking_above` instead.

In [ ]:
import json, time
from concurrent.futures import ProcessPoolExecutor, as_completed


def done_keys(path):
    '''Keys already in the output, so a rerun extends rather than repeats.'''
    if not os.path.exists(path):
        return set()
    keys = set()
    with open(path) as handle:
        for line in handle:
            try:
                keys.add(json.loads(line)["key"])
            except Exception:
                pass
    return keys


def run(jobs, out_path, workers):
    '''Measure every board not already recorded, in parallel, appending as it goes.'''
    workers = max(1, min(workers, 61))  # the Windows ceiling, harmless elsewhere
    seen = done_keys(out_path)
    pending = [j for j in jobs if j[0] not in seen]
    if seen:
        print("skipping %d results already in %s" % (len(seen), out_path))
    print("%d boards to measure on %d workers" % (len(pending), workers))
    if CONFIG["timeout"] and not W.CAN_TIME_OUT:
        print("this platform has no SIGALRM, so the per-board timeout is not enforced; "
              "cap the work with skip_marking_above and max_cells instead")

    started = time.time()
    done = 0
    with open(out_path, "a") as handle:
        def record(result):
            handle.write(json.dumps(result) + "\n")
            handle.flush()
            note = "%dx%d M=%d k=%d " % (result["rows"], result["cols"],
                                         result["solutions"], result["rounds"])
            if "error" in result:
                note += "ERROR " + result["error"]
            elif "timeout" in result:
                note += "timed out after %ss" % result["timeout"]
            else:
                note += "clue 2q=%(two_qubit)s depth=%(depth)s" % result["clue"]
                if "marking" in result:
                    note += " | marking 2q=%s (%sx)" % (result["marking"]["two_qubit"],
                                                        result["ratio_two_qubit"])
            print("[%d/%d] %s  %.0fs" % (done, len(pending), note, time.time() - started),
                  flush=True)

        if workers > 1 and pending:
            # max_tasks_per_child keeps a long sweep from inheriting whatever one large
            # transpile left behind in a worker. It needs Python 3.11; without it the pool
            # simply keeps its workers.
            try:
                pool = ProcessPoolExecutor(max_workers=workers, max_tasks_per_child=1)
            except (TypeError, ValueError):
                pool = ProcessPoolExecutor(max_workers=workers)
            with pool:
                futures = [pool.submit(W.measure_board, job) for job in pending]
                for future in as_completed(futures):
                    done += 1
                    record(future.result())
        else:
            for job in pending:
                done += 1
                record(W.measure_board(job))

    print("done in %.0fs" % (time.time() - started))

## 7. Question 1 — does the ratio hold as the board grows?

One board per (shape, solution count), smallest shapes first. Transposes are dropped: the
circuit depends on the cell count and the solution count, not on the aspect ratio.

In [ ]:
def shapes_up_to(min_cells, max_cells):
    out = [(r, c) for r in range(2, max_cells + 1) for c in range(r, max_cells + 1)
           if min_cells <= r * c <= max_cells]
    return sorted(out, key=lambda rc: (rc[0] * rc[1], rc))


def scaling_jobs():
    for rows, cols in shapes_up_to(SCALING["min_cells"], SCALING["max_cells"]):
        per_count = {}
        for row_clues, col_clues, solutions in W.boards(rows, cols, None, CONFIG["sample_seed"]):
            if solutions in per_count or solutions > SCALING["max_solutions"]:
                continue
            per_count[solutions] = True
            yield ("scaling:%dx%d:M%d" % (rows, cols, solutions),
                   row_clues, col_clues, solutions, CONFIG)
            if len(per_count) >= SCALING["counts_per_shape"]:
                break

print("shapes:", shapes_up_to(SCALING["min_cells"], SCALING["max_cells"]))

In [ ]:
run(list(scaling_jobs()), OUT_SCALING, CONFIG["workers"])

## 8. Question 2 — what does the spread look like?

Every solvable board of one shape, or a sample when the space is too large to enumerate.
Change `SPREAD["shape"]` and rerun to cover another size; the output file accumulates.

In [ ]:
def spread_jobs():
    rows, cols = SPREAD["shape"]
    for index, (row_clues, col_clues, solutions) in enumerate(
            W.boards(rows, cols, SPREAD["sample"], CONFIG["sample_seed"])):
        yield ("spread:%dx%d:%d" % (rows, cols, index), row_clues, col_clues, solutions, CONFIG)

run(list(spread_jobs()), OUT_SPREAD, CONFIG["workers"])

## 9. Read the results

In [ ]:
import statistics

def load(path):
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path) as handle:
        for line in handle:
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows


def summarize(path):
    rows = load(path)
    usable = [r for r in rows if "clue" in r]
    print("%s: %d measured, %d failed or timed out\n" % (path, len(usable), len(rows) - len(usable)))
    groups = {}
    for r in usable:
        groups.setdefault((r["cells"], "%dx%d" % (r["rows"], r["cols"]), r["solutions"]), []).append(r)

    print("%7s %3s %4s %5s %28s %12s %10s" %
          ("shape", "M", "n", "runs", "clue 2q min/med/max", "marking 2q", "ratio med"))
    for (cells, shape, solutions), group in sorted(groups.items()):
        clue = sorted(r["clue"]["two_qubit"] for r in group)
        marking = [r["marking"]["two_qubit"] for r in group if "marking" in r]
        ratios = [r["ratio_two_qubit"] for r in group if "ratio_two_qubit" in r]
        spread = "%s / %s / %s" % (format(clue[0], ","), format(clue[len(clue) // 2], ","),
                                   format(clue[-1], ","))
        print("%7s %3d %4d %5d %28s %12s %10s" % (
            shape, solutions, cells, len(group), spread,
            format(int(statistics.median(marking)), ",") if marking else "-",
            ("%.2fx" % statistics.median(ratios)) if ratios else "-"))

summarize(OUT_SCALING)
print()
summarize(OUT_SPREAD)

In [ ]:
# A quick look at the spread, if matplotlib is to hand. Nothing downstream needs it.
try:
    import matplotlib.pyplot as plt

    rows = [r for r in load(OUT_SPREAD) if "clue" in r]
    if rows:
        by_m = {}
        for r in rows:
            by_m.setdefault(r["solutions"], []).append(r["clue"]["two_qubit"])
        keys = sorted(by_m)[:6]
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.boxplot([by_m[k] for k in keys], labels=["M=%d" % k for k in keys])
        ax.set_yscale("log")
        ax.set_ylabel("two-qubit gates, clue oracle")
        ax.set_title("%dx%d, %d boards" % (SPREAD["shape"][0], SPREAD["shape"][1], len(rows)))
        plt.tight_layout()
        plt.show()
    else:
        print("no spread results yet")
except ImportError:
    print("matplotlib not installed; skipping the plot")

## 10. What to send back

Both `.jsonl` files. Every record carries its own clue pair, solution count, round count,
seeds and settings, so the results can be folded in without re-deriving anything or
guessing what configuration produced them.

Worth noting when you report: whether `CONFIG["device"]` was on, what `seeds` was, and
which shapes hit `skip_marking_above` so their ratio column is absent rather than estimated.